# Lab 1 — Q-Learning



### Import libraries

In [1]:
import numpy as np
import random

###  Define states and reward matrix
- According to the environment in your lab, we have states 0 to 5, with state 5 as the goal.
- The reward matrix can be defined as follows:

In [2]:
# Define the states
states = [0, 1, 2, 3, 4, 5]

# Define the reward matrix
# -1  = invalid transition
#  0  = valid transition
# 100 = reaching the goal state

R = np.array([
    [-1, -1, -1, -1,  0, -1],   # State 0
    [-1, -1, -1,  0, -1, 100],   # State 1
    [-1, -1, -1,  0, -1, -1],   # State 2
    [-1,  0,  0, -1,  0, 100],   # State 3
    [ 0, -1, -1,  0, -1, 100],   # State 4
    [-1,  0, -1, -1,  0, 100]    # State 5
])

print("States:")
print(states)

print("\nReward Matrix:")
print(R)

States:
[0, 1, 2, 3, 4, 5]

Reward Matrix:
[[ -1  -1  -1  -1   0  -1]
 [ -1  -1  -1   0  -1 100]
 [ -1  -1  -1   0  -1  -1]
 [ -1   0   0  -1   0 100]
 [  0  -1  -1   0  -1 100]
 [ -1   0  -1  -1   0 100]]


### What this means

For example:

```python
R[0][4] = 0
This means the agent can move from state 0 to state 4.
And:
R[4][5] = 100
This means the agent can move from state 4 to the goal state 5 and receives a reward of 100.
Invalid movements have a reward of -1.
This matches the connections represented in the sample environment and expected Q-table in the lab. The expected output shows, for example, state 0 connected to 4, state 2 connected to 3, and states 1, 3, and 4 connected toward the goal.


### Initialize Q-table
- Your lab specifically asks for the Q-table to be initialized with zeros.

In [3]:
# Initialize Q-table with zeros
Q = np.zeros((6, 6))

print("Initial Q-Table:")
print(Q)

Initial Q-Table:
[[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]


### Define valid actions
#### This is one of the requirements in your lab:
- Select valid actions from the current state.

In [4]:
def get_valid_actions(state):
    return np.where(R[state] >= 0)[0]

In [5]:
print("Valid actions from state 0:", get_valid_actions(0))
print("Valid actions from state 3:", get_valid_actions(3))

Valid actions from state 0: [4]
Valid actions from state 3: [1 2 4 5]


Here, an **action is represented by the next state**.

So, if the current state is **3** and the action is **5**, the agent moves:

**3 → 5**

and receives a reward of **100**.

###  Q-Learning parameters

In [6]:
# Q-Learning parameters
alpha = 0.8       # Learning rate
gamma = 0.8       # Discount factor
episodes = 1000   # Number of training episodes

goal_state = 5

I am using **gamma = 0.8** because it produces the type of values shown in the lab's expected output.

For example:

**Path:**

`0 → 4 → 5`

gives approximately:

$$
Q(0,4) = 0.8 \times 100 = 80
$$

And:

**Path:**

`2 → 3 → 5`

gives approximately:

$$
Q(2,3) = 0.8 \times 100 \times 0.8 = 64
$$

These values match the values shown in the **expected Q-table** in the lab document.

###  Train the Q-Learning agent


In [7]:
# Train the agent
for episode in range(episodes):

    # Start from a random state other than the goal
    current_state = random.randint(0, 4)

    while current_state != goal_state:

        # Select valid actions
        valid_actions = get_valid_actions(current_state)

        # Select a random valid action
        next_state = random.choice(valid_actions)

        # Get the reward
        reward = R[current_state, next_state]

        # Get valid actions from the next state
        next_valid_actions = get_valid_actions(next_state)

        # Find the maximum Q-value of the next state
        if len(next_valid_actions) > 0:
            max_next_q = np.max(Q[next_state, next_valid_actions])
        else:
            max_next_q = 0

        # Q-Learning update
        Q[current_state, next_state] = Q[current_state, next_state] + alpha * (
            reward
            + gamma * max_next_q
            - Q[current_state, next_state]
        )

        # Move to the next state
        current_state = next_state

The important formula
The code implements:
$$ Q(s,a) = Q(s,a) + \alpha \left[ R(s,a) + \gamma \max Q(s',a') - Q(s,a) \right] $$
This is the actual Q-Learning update your faculty will expect you to understand.

### Display the learned Q-table

In [8]:
print("Learned Q-Table:")
print(np.round(Q, 2))

Learned Q-Table:
[[  0.   0.   0.   0.  80.   0.]
 [  0.   0.   0.  80.   0. 100.]
 [  0.   0.   0.  80.   0.   0.]
 [  0.  80.  64.   0.  80. 100.]
 [ 64.   0.   0.  80.   0. 100.]
 [  0.   0.   0.   0.   0.   0.]]


###  Find the optimal path

In [9]:
def find_optimal_path(start_state):

    path = [start_state]
    current_state = start_state

    while current_state != goal_state:

        # Find valid actions
        valid_actions = get_valid_actions(current_state)

        # Select the action with the highest Q-value
        next_state = valid_actions[
            np.argmax(Q[current_state, valid_actions])
        ]

        path.append(next_state)
        current_state = next_state

    return path


### Test with starting state 2

In [10]:
start_state = 2

optimal_path = find_optimal_path(start_state)

print("Starting State:", start_state)
print("Goal State:", goal_state)
print("Optimal Path:", " -> ".join(map(str, optimal_path)))

Starting State: 2
Goal State: 5
Optimal Path: 2 -> 3 -> 5


## Learning

- Q-Learning is a reinforcement learning algorithm that learns the best action to take in each state by interacting with the environment.
- The Q-table is initially initialized with zeros and is updated during each training episode based on the reward received and the estimated future reward.
- The **learning rate (α)** controls how much the new information affects the existing Q-value.
- The **discount factor (γ)** determines the importance of future rewards.
- Valid actions are selected based on the connections available in the reward matrix.
- After sufficient training, the Q-table contains higher Q-values for actions that lead toward the goal state.
- The optimal path can be obtained by selecting the valid action with the highest Q-value at each state.

## Conclusion

The Q-Learning algorithm was successfully implemented for the given environment containing six states, with **state 5 as the goal state**. The agent learned the values of different state-action pairs through repeated training episodes. The learned Q-table was used to determine the optimal path from the given starting state to the goal state.

For the given starting state **2**, the learned optimal path is:

**2 → 3 → 5**

Thus, the experiment demonstrates how Q-Learning can learn an optimal sequence of actions through rewards and interaction with the environment.